# 6CS012 - Worksheet 4
## Building a Fully Connected Neural Network for Devnagari Handwritten Digit Classification

This notebook covers all 7 tasks:
1. Data Preparation
2. Build the FCN Model
3. Compile the Model
4. Train the Model
5. Evaluate the Model
6. Save and Load the Model
7. Making Predictions

---
## Task 1: Data Preparation
Load images from the dataset folder using PIL, normalize them, and one-hot encode the labels.

In [ ]:
# Install / import required libraries
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from PIL import Image

print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTANT: Upload your dataset zip file to Colab first, then
# run this cell to unzip it.
# Expected structure after unzip:
#   dataset/
#     Train/  digit_0/ ... digit_9/
#     Test/   digit_0/ ... digit_9/
# ─────────────────────────────────────────────────────────────

# If your dataset is a zip file, uncomment and run the lines below:
# from google.colab import files
# uploaded = files.upload()           # choose your zip file
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
#     z.extractall('.')               # extracts to current directory

# Verify the folder structure exists
train_dir = "dataset/Train/"
test_dir  = "dataset/Test/"

assert os.path.exists(train_dir), f"Train folder not found at: {train_dir}"
assert os.path.exists(test_dir),  f"Test folder not found at:  {test_dir}"

print("Dataset folders found!")
print("Classes in Train:", sorted(os.listdir(train_dir)))

In [ ]:
# ── Image settings ──────────────────────────────────────────
IMG_HEIGHT, IMG_WIDTH = 28, 28
NUM_CLASSES = 10

def load_images_from_folder(folder):
    """
    Loads all images from a folder structured as:
        folder/class_name/image_file
    Returns:
        images : np.ndarray  shape (N, 28, 28, 1), normalised to [0,1]
        labels : np.ndarray  shape (N, NUM_CLASSES), one-hot encoded
    """
    images = []
    labels = []

    # Sort so class index is deterministic: digit_0 → 0, digit_1 → 1, …
    class_names = sorted(os.listdir(folder))
    class_map   = {name: i for i, name in enumerate(class_names)}
    print(f"Found {len(class_names)} classes: {class_names}")

    for class_name in class_names:
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue
        label = class_map[class_name]

        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)
            try:
                img = Image.open(img_path).convert("L")            # grayscale
                img = img.resize((IMG_WIDTH, IMG_HEIGHT))           # 28×28
                img_array = np.array(img) / 255.0                  # normalise
                images.append(img_array)
                labels.append(label)
            except Exception as e:
                print(f"  Skipping {img_path}: {e}")

    images = np.array(images, dtype=np.float32).reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)
    labels = to_categorical(np.array(labels), num_classes=NUM_CLASSES)
    return images, labels


# Load datasets
print("Loading training data …")
x_train, y_train = load_images_from_folder(train_dir)

print("\nLoading test data …")
x_test, y_test = load_images_from_folder(test_dir)

print(f"\nTraining set : {x_train.shape}  |  Labels: {y_train.shape}")
print(f"Test set     : {x_test.shape}   |  Labels: {y_test.shape}")

In [ ]:
# Visualise 10 sample training images
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(IMG_HEIGHT, IMG_WIDTH), cmap='gray')
    plt.title(f"Label: {np.argmax(y_train[i])}")
    plt.axis('off')
plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

---
## Task 2: Build the FCN Model
Sequential model with 3 hidden layers (64 → 128 → 256 neurons, sigmoid activation) and a softmax output layer.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_fcn_model():
    model = keras.Sequential([
        # Input + Flatten: 28×28×1 → 784-D vector
        keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
        keras.layers.Flatten(),

        # Hidden Layer 1
        keras.layers.Dense(64,  activation='sigmoid', name='hidden_1'),

        # Hidden Layer 2
        keras.layers.Dense(128, activation='sigmoid', name='hidden_2'),

        # Hidden Layer 3
        keras.layers.Dense(256, activation='sigmoid', name='hidden_3'),

        # Output Layer — 10 classes, softmax for probability distribution
        keras.layers.Dense(NUM_CLASSES, activation='softmax', name='output'),
    ], name='Devnagari_FCN')
    return model

model = build_fcn_model()
model.summary()

---
## Task 3: Compile the Model
Use Adam optimizer, categorical cross-entropy loss, and accuracy metric.

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',   # y_train is one-hot encoded
    metrics=['accuracy']
)

print("Model compiled successfully!")
print(f"  Optimizer : adam")
print(f"  Loss      : categorical_crossentropy")
print(f"  Metrics   : accuracy")

---
## Task 4: Train the Model
Batch size = 128, epochs = 20, validation split = 0.2. ModelCheckpoint and EarlyStopping callbacks included.

In [ ]:
BATCH_SIZE = 128
EPOCHS     = 20

callbacks = [
    # Save the best model (lowest val_loss) during training
    keras.callbacks.ModelCheckpoint(
        filepath='best_devnagari_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    # Stop early if val_loss stops improving for 5 consecutive epochs
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
]

history = model.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,       # 20% of training data used for validation
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# ── Plot Training & Validation Loss and Accuracy ─────────────
train_loss = history.history['loss']
val_loss   = history.history['val_loss']
train_acc  = history.history['accuracy']
val_acc    = history.history['val_accuracy']
epochs_ran = range(1, len(train_loss) + 1)

plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(epochs_ran, train_loss, 'b-o', label='Training Loss')
plt.plot(epochs_ran, val_loss,   'r-o', label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(epochs_ran, train_acc, 'b-o', label='Training Accuracy')
plt.plot(epochs_ran, val_acc,   'r-o', label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.suptitle('Model Training Progress', fontsize=15)
plt.tight_layout()
plt.show()

---
## Task 5: Evaluate the Model
Use `model.evaluate()` on the test set to measure loss and accuracy on unseen data.

In [ ]:
print("Evaluating model on test set …\n")
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)

print(f"\n{'='*35}")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_acc * 100:.2f}%")
print(f"{'='*35}")

---
## Task 6: Save and Load the Model
Save the trained model to an `.h5` file and reload it to confirm it works correctly.

In [ ]:
# ── Save ────────────────────────────────────────────────────
model.save('devnagari_fcn_model.h5')
print("Model saved to 'devnagari_fcn_model.h5'")

In [ ]:
# ── Load & Re-evaluate ──────────────────────────────────────
loaded_model = tf.keras.models.load_model('devnagari_fcn_model.h5')
print("Model loaded successfully!\n")

loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test, verbose=2)
print(f"\nLoaded Model — Test Loss: {loaded_loss:.4f} | Test Accuracy: {loaded_acc * 100:.2f}%")

---
## Task 7: Making Predictions
Use `model.predict()` to get class probabilities, then convert to digit labels with `np.argmax()`.

In [ ]:
# Get predictions (probabilities) for the entire test set
predictions = model.predict(x_test, verbose=0)

# Convert probability arrays to class labels
predicted_labels = np.argmax(predictions, axis=1)   # predicted class
true_labels      = np.argmax(y_test,      axis=1)   # actual class

# Quick spot-check: first 5 samples
print("Sample predictions (first 5 test images):")
print(f"{'Index':<8} {'True Label':<15} {'Predicted Label':<15} {'Correct?'}")
print("-" * 50)
for i in range(5):
    correct = "✓" if true_labels[i] == predicted_labels[i] else "✗"
    print(f"{i:<8} {true_labels[i]:<15} {predicted_labels[i]:<15} {correct}")

In [ ]:
# Visualise 10 test images with their true and predicted labels
plt.figure(figsize=(14, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].reshape(IMG_HEIGHT, IMG_WIDTH), cmap='gray')
    color = 'green' if predicted_labels[i] == true_labels[i] else 'red'
    plt.title(f"True: {true_labels[i]}\nPred: {predicted_labels[i]}",
              color=color, fontsize=9)
    plt.axis('off')
plt.suptitle('Test Predictions  (green = correct, red = wrong)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for a full picture of per-class performance
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(true_labels, predicted_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=[f"Digit {i}" for i in range(NUM_CLASSES)])

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
plt.title('Confusion Matrix — Devnagari Digit Classification', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| Task 1 | Loaded Devnagari images with PIL, normalised and one-hot encoded labels |
| Task 2 | Built a Sequential FCN: Flatten → 64 → 128 → 256 (sigmoid) → 10 (softmax) |
| Task 3 | Compiled with Adam optimiser and categorical cross-entropy loss |
| Task 4 | Trained for up to 20 epochs (batch=128, val_split=0.2) with EarlyStopping |
| Task 5 | Evaluated on the test set — printed loss and accuracy |
| Task 6 | Saved model as `.h5`, reloaded and re-evaluated |
| Task 7 | Made predictions, visualised results, plotted confusion matrix |